# QuantumSight — Hybrid Quantum-Classical Image Classification

This notebook provides an interactive walkthrough of the QuantumSight architecture.
We demonstrate:
1. Loading experiment configuration (`configs.config`)
2. Inspecting the Quantum Circuit components (`src.quantum`)
3. Initializing Classical Baseline and Hybrid Quantum-Classical models (`src.models`)
4. Running forward pass and tracking resources (`src.utils`)
5. Visualizing feature representations and circuit layouts (`src.visualization`)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import pennylane as qml
from configs.config import DEVICE, EXPERIMENTS, DATASETS
from src.quantum import get_quantum_device, amplitude_circuit, angle_circuit, CIRCUITS
from src.models import EnhancedClassicalCNN, EnhancedHybridModel
from src.utils import ResourceTracker
from src.visualization import MLVisualizer

print(f"Compute Device: {DEVICE}")
print("Available Experiments:", list(EXPERIMENTS.keys()))

## 1. Inspect Model Architecture
We instantiate both the Classical CNN baseline and the Hybrid Quantum-Classical model for MNIST.

In [ ]:
cfg = EXPERIMENTS["mnist_4q"]
hybrid_model = EnhancedHybridModel(
    "MNIST",
    num_qubits=cfg["num_qubits"],
    num_layers=cfg["num_layers"],
    encoding=cfg["encoding"]
).to(DEVICE)

classical_model = EnhancedClassicalCNN("MNIST").to(DEVICE)

h_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
c_params = sum(p.numel() for p in classical_model.parameters() if p.requires_grad)

print(f"Hybrid Trainable Parameters:    {h_params}")
print(f"Classical Trainable Parameters: {c_params}")

## 2. Parameterized Quantum Circuit Visualization

In [ ]:
dummy_weights = torch.randn((cfg["num_layers"], cfg["num_qubits"]))
circuit_fn = CIRCUITS[cfg["encoding"]]
dummy_input = torch.zeros(2 ** cfg["num_qubits"])

print(qml.draw(circuit_fn)(dummy_input, dummy_weights))